In [2]:
import os
import json
import random
import shutil
import cv2
import yaml
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

# ==========================================
# 1. ORTAM ALGILAMA (Colab vs. Lokal PC)
# ==========================================
IN_COLAB = "COLAB_GPU" in os.environ or os.path.exists("/content")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path("/content/drive/MyDrive/training_data/quadrant")
else:
    # görevler/ klasörünün bir üstündeki training_data/quadrant klasörünü bulur
    BASE_DIR = Path("../training_data/quadrant").resolve()

print("Aktif Çalışma Dizini:", BASE_DIR)

# ==========================================
# 2. COCO JSON -> YOLO ETİKET DÖNÜŞÜMÜ
# ==========================================
coco_path = BASE_DIR / "train_quadrant.json"
labels_dir = BASE_DIR / "labels"
labels_dir.mkdir(parents=True, exist_ok=True)

with open(coco_path, "r") as f:
    data = json.load(f)

images_info = {
    img["id"]: (Path(img["file_name"]).stem, img["width"], img["height"])
    for img in data["images"]
}

categories = sorted(data["categories"], key=lambda x: x["id"])
cat_id_map = {cat["id"]: idx for idx, cat in enumerate(categories)}
yolo_labels = {stem: [] for stem, _, _ in images_info.values()}

for ann in data["annotations"]:
    img_id = ann["image_id"]
    if img_id not in images_info:
        continue
    stem, img_w, img_h = images_info[img_id]
    raw_cat_id = ann.get("category_id_1", ann.get("category_id"))
    cls_idx = cat_id_map[raw_cat_id]

    x_min, y_min, w, h = ann["bbox"]
    x_center = max(0.0, min(1.0, (x_min + w / 2.0) / img_w))
    y_center = max(0.0, min(1.0, (y_min + h / 2.0) / img_h))
    w_norm = max(0.0, min(1.0, w / img_w))
    h_norm = max(0.0, min(1.0, h / img_h))

    yolo_labels[stem].append(f"{cls_idx} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}")

for stem, lines in yolo_labels.items():
    with open(labels_dir / f"{stem}.txt", "w") as f:
        f.write("\n".join(lines))

print(f"Etiketler hazır: {len(yolo_labels)} dosya")

# ==========================================
# 3. VERİ BÖLME (70/15/15) + OPENCV CLAHE ÖN İŞLEME
# ==========================================
def save_with_clahe(src_path, dst_path):
    """Röntgen kontrastını iyileştirmek için OpenCV CLAHE uygular."""
    img = cv2.imread(str(src_path), cv2.IMREAD_GRAYSCALE)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(img)
    enhanced_bgr = cv2.cvtColor(enhanced, cv2.COLOR_GRAY2BGR)
    cv2.imwrite(str(dst_path), enhanced_bgr)

images_src = BASE_DIR / "xrays"
image_files = sorted([f for f in images_src.glob("*.*") if f.suffix.lower() in [".png", ".jpg", ".jpeg"]])

# Sena ile birebir aynı test seti üzerinde adil kıyaslama için seed=42
random.seed(42)
random.shuffle(image_files)

n_total = len(image_files)
n_train = int(n_total * 0.70)
n_val = int(n_total * 0.15)

splits = {
    "train": image_files[:n_train],
    "val": image_files[n_train:n_train + n_val],
    "test": image_files[n_train + n_val:]
}

for split_name, file_list in splits.items():
    img_dst = BASE_DIR / "images" / split_name
    lbl_dst = BASE_DIR / "labels" / split_name
    img_dst.mkdir(parents=True, exist_ok=True)
    lbl_dst.mkdir(parents=True, exist_ok=True)

    for img_path in file_list:
        save_with_clahe(img_path, img_dst / img_path.name)
        lbl_path = labels_dir / f"{img_path.stem}.txt"
        if lbl_path.exists():
            shutil.copy(lbl_path, lbl_dst / lbl_path.name)

for cache_file in labels_dir.glob("*.cache"):
    cache_file.unlink()

print(f"CLAHE uygulandı ve bölündü -> Train: {len(splits['train'])} | Val: {len(splits['val'])} | Test: {len(splits['test'])}")

# ==========================================
# 4. YAML KONFİGÜRASYONU
# ==========================================
cats = {idx: str(cat["name"]) for idx, cat in enumerate(categories)}
dataset = {
    "path": str(BASE_DIR.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": cats
}

yaml_path = BASE_DIR / "quadrant_opt.yaml"
with open(yaml_path, "w") as f:
    yaml.dump(dataset, f, default_flow_style=False, sort_keys=False)

Aktif Çalışma Dizini: C:\Users\ceyda\OneDrive\Masaüstü\26-27 GÜZ\Ara Proje\Git\panoramik-rontgenlerde-dis-sayimi-anomali-tespiti\training_data\quadrant
Etiketler hazır: 693 dosya
CLAHE uygulandı ve bölündü -> Train: 485 | Val: 103 | Test: 105


In [ ]:
# ==========================================
# 5. OPTİMİZE EDİLMİŞ MODEL EĞİTİMİ
# ==========================================
selected_device = "0" if torch.cuda.is_available() else "cpu"
print(f"Eğitim cihazı: {selected_device}")

model = YOLO("yolov8n.pt")
results = model.train(
    data=str(yaml_path),
    epochs=50,               # 10 yerine 50 epoch (kutu sınırlarının oturması için)
    patience=15,             # 15 epoch iyileşme olmazsa erken durdur
    batch=8,                 # Küçük veri setinde daha dengeli güncelleme
    imgsz=640,
    optimizer="AdamW",       # Daha kararlı yakınsama
    lr0=0.001,
    cos_lr=True,
    fliplr=0.0,              # Sağ/sol kadran yönünü koru
    mosaic=0.0,              # Anatomik yapıyı bozan mosaic artırımını kapat
    hsv_h=0.0,               # Gri röntgende yapay renk oynamalarını kapat
    hsv_s=0.0,
    device=selected_device,
    project="dentex_quadrant_opt",
    name="clahe_adamw_run1"
)

# ==========================================
# 6. TEST SETİ DEĞERLENDİRMESİ
# ==========================================
best_weights = Path(results.save_dir) / "weights" / "best.pt"
best_model = YOLO(str(best_weights))
test_metrics = best_model.val(data=str(yaml_path), split="test")

print("\n--- OPTİMİZE MODEL TEST SONUÇLARI ---")
print(f"Test mAP50:    {test_metrics.box.map50:.4f} (Baseline: 0.9950)")
print(f"Test mAP50-95: {test_metrics.box.map:.4f} (Baseline: 0.7122)")
print(f"Test Precision:{test_metrics.box.mp:.4f}")
print(f"Test Recall:   {test_metrics.box.mr:.4f}")